# Multi-GPU HEALPix transforms with the s2fft CUDA backend

Sharded `s2fft.forward` (map to harmonics) over a JAX device mesh, comparing the
`jax_cuda` custom-CUDA path against the pure-JAX path. Uses only `s2fft` — no `jax_healpy`.

## Before running on the cluster

This notebook requires the **ASKabalan fork build** of `s2fft` (per-device cuFFT plan
cache + stream pools, fixed FFI aliasing; see `HANDOFF.md`). The verification cell asserts
the fork version and runs a single-GPU `jax_cuda` sanity check. If the cluster venv holds
the upstream package, install the fork first (wheel/git — delivery of the build to the
cluster is handled outside this notebook).

If cuFFT plan creation fails with code 5 on small GPUs, cap JAX's pool:
`XLA_PYCLIENT_MEM_FRACTION<=0.8` (export before starting the kernel).

In [1]:
from functools import partial

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
from jax.sharding import AxisType, NamedSharding
from jax.sharding import PartitionSpec as P

import s2fft
from s2fft.utils import signal_generator

jax.config.update("jax_enable_x64", True)

nside = 32
L = 2 * nside
key = jr.key(0)

mesh = jax.make_mesh((jax.device_count(), 1), ("x", "y"), axis_types=(AxisType.Auto, AxisType.Auto))
sharding = NamedSharding(mesh, P("x", "y"))
print(f"Mesh shape     : {mesh.shape}")
print(f"Partition spec : {sharding.spec}")

JAX is not using 64-bit precision. This will dramatically affect numerical precision at even moderate L.


Mesh shape     : OrderedDict({'x': 4, 'y': 1})
Partition spec : P('x', 'y')


In [2]:
# Environment verification: this notebook requires the ASKabalan fork build of s2fft
# (per-device cuFFT plan cache + stream pools, fixed FFI aliasing). If this cell fails,
# the venv still holds the upstream package and every jax_cuda cell below is void.
import importlib.metadata as md

import s2fft_lib._s2fft as _ext

version = md.version("s2fft")
assert version.startswith("1.4.1.dev"), f"upstream s2fft {version} installed - install the fork build first"
assert _ext.COMPILED_WITH_CUDA, "s2fft_lib was compiled without CUDA support"
print(f"s2fft {version} | CUDA extension OK | jax {jax.__version__} | devices: {jax.devices()}")

# Tiny single-GPU sanity check through the CUDA path (agrees with the jax path).
nside_s = 8
L_s = 2 * nside_s
rng = np.random.default_rng(0)
flm_s = signal_generator.generate_flm(rng=rng, L=L_s, reality=True)
f_s = s2fft.inverse(flm_s, L=L_s, nside=nside_s, sampling="healpix", reality=True, method="jax")
alm_c = s2fft.forward(f_s, L=L_s, nside=nside_s, sampling="healpix", reality=True, method="jax_cuda")
alm_j = s2fft.forward(f_s, L=L_s, nside=nside_s, sampling="healpix", reality=True, method="jax")
rel = float(jnp.max(jnp.abs(alm_c - alm_j)) / jnp.max(jnp.abs(alm_j)))
print(f"single-GPU forward sanity: max|a_cuda-a_jax|/max|a_jax| = {rel:.2e}")
assert rel < 1e-10, "jax_cuda disagrees with jax on a single GPU"

s2fft 1.4.1.dev21 | CUDA extension OK | jax 0.11.1 | devices: [CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2), CudaDevice(id=3)]
single-GPU forward sanity: max|a_cuda-a_jax|/max|a_jax| = 1.55e-16


In [23]:
npix = 12 * nside**2

# One random real sky per device, generated through the jax path.
flm_per_device = [
    signal_generator.generate_flm(rng=np.random.default_rng(d), L=L, reality=True)
    for d in range(jax.device_count())
]
f_per_device = [
    s2fft.inverse(flm, L=L, nside=nside, sampling="healpix", reality=True, method="jax")
    for flm in flm_per_device
]
hp_map = jnp.asarray(np.stack(f_per_device))
hp_map = jax.lax.with_sharding_constraint(hp_map, sharding)


def sharded_forward(f, method):
    def inner_fn(f):
        return s2fft.forward(
            f.squeeze(),
            L=L,
            nside=nside,
            sampling="healpix",
            reality=False,
            method=method,
        )
    return jax.shard_map(inner_fn, mesh=mesh, in_specs=P("x", None), out_specs=P("x", None))(f)

jax.debug.visualize_array_sharding(hp_map)

                                                                                
                                     GPU 0                                      
                                                                                
                                                                                
                                     GPU 1                                      
                                                                                
                                                                                
                                     GPU 2                                      
                                                                                
                                                                                
                                     GPU 3                                      
                                                                                

In [24]:
jax.clear_caches()

In [25]:
%time cuda_flm = sharded_forward(hp_map, method="jax_cuda").block_until_ready()

CPU times: user 2.73 s, sys: 97.7 ms, total: 2.83 s
Wall time: 1.67 s


In [26]:
%time jax_flm = sharded_forward(hp_map, method="jax").block_until_ready()

CPU times: user 10.5 s, sys: 358 ms, total: 10.8 s
Wall time: 4.98 s


In [27]:
# Determinism: two independent jax_cuda sharded runs must agree bit-for-bit.
cuda_flm_1 = sharded_forward(hp_map, method="jax_cuda").block_until_ready()
cuda_flm_2 = sharded_forward(hp_map, method="jax_cuda").block_until_ready()
max_diff = float(jnp.max(jnp.abs(cuda_flm_1 - cuda_flm_2)))
print(f"jax_cuda determinism across runs: max|run1 - run2| = {max_diff:.3e}")
assert max_diff == 0.0, "jax_cuda sharded forward is not deterministic"

jax_cuda determinism across runs: max|run1 - run2| = 0.000e+00


In [28]:
# Agreement: sharded jax_cuda vs sharded jax on identical data (float64).
rel = float(jnp.max(jnp.abs(cuda_flm_1 - jax_flm)) / jnp.max(jnp.abs(jax_flm)))
print(f"sharded jax_cuda vs jax: max|a_cuda-a_jax|/max|a_jax| = {rel:.2e}")
assert rel < 1e-10, "sharded jax_cuda disagrees with sharded jax"

sharded jax_cuda vs jax: max|a_cuda-a_jax|/max|a_jax| = 3.34e-16


In [29]:
%timeit cuda_flm = sharded_forward(hp_map, method="jax_cuda").block_until_ready()

1.23 s ± 19.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [30]:
%timeit jax_flm = sharded_forward(hp_map, method="jax").block_until_ready()

4.49 s ± 22.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
print("Finished all good")

Finished all good
